# 03. Data Analysis & Clustering (PySpark)

Notebook ini mengikuti format bertahap seperti notebook preprocessing. Isi notebook dimulai dari membaca data bersih dari MongoDB, lalu analisis statistik deskriptif, visualisasi persebaran data, penentuan jumlah cluster optimal, pelatihan model clustering, evaluasi Silhouette Score, hingga penyimpanan hasil cluster untuk interpretasi zona rawan gempa.

### Tahap 1: Import Library & Load Konfigurasi
Menyiapkan environment PySpark, mengaktifkan library eksternal (pandas, matplotlib, seaborn), serta mengambil *environment variables*.

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, avg, min, max, stddev_samp, round, date_format, rand, row_number
from pyspark.ml.functions import array_to_vector
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator

load_dotenv('../.env')
SPARK_MASTER = os.getenv('SPARK_MASTER_URL', 'local[*]')
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'earthquake_db')
MONGO_CLEAN_COL = os.getenv('MONGO_CLEAN_COLLECTION', 'clean_earthquakes')
MONGO_EDA_SUMMARY_COL = os.getenv('MONGO_EDA_SUMMARY_COLLECTION', 'eda_summary')
MONGO_MODEL_METRICS_COL = os.getenv('MONGO_MODEL_METRICS_COLLECTION', 'model_metrics')
MONGO_KMEANS_COL = os.getenv('MONGO_KMEANS_COLLECTION', 'kmeans_results')
MONGO_BISECTING_COL = os.getenv('MONGO_BISECTING_COLLECTION', 'bisecting_results')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

### Tahap 2: Menyalakan Spark Session & Membaca Data Bersih
Membuka sesi di atas master PySpark dan mengambil data yang sudah berada di koleksi `clean_earthquakes`.

In [ ]:
spark = SparkSession.builder \
    .appName('EarthquakeAnalysis') \
    .master(SPARK_MASTER) \
    .config('spark.mongodb.read.connection.uri', MONGO_URI) \
    .config('spark.mongodb.write.connection.uri', MONGO_URI) \
    .config('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:10.3.0') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark Session aktif di master: {SPARK_MASTER}')

In [ ]:
df_clean = spark.read.format('mongodb') \
    .option('database', MONGO_DB) \
    .option('collection', MONGO_CLEAN_COL) \
    .load() \
    .filter(col('mag') >= 2.5)

print(f'Total data bersih (Filter Mag >= 2.5): {df_clean.count()} baris')
df_clean.printSchema()

df_clean = df_clean.withColumn('features', array_to_vector(col('scaled_features')))
df_clean.createOrReplaceTempView('earthquake_clean')

df_clean.select('time', 'country', 'latitude', 'longitude', 'depth', 'mag').show(5, truncate=False)

### Tahap 3: Analisis Statistik Deskriptif
Meninjau karakteristik dasar data gempa yang sudah bersih sebelum masuk ke visualisasi dan clustering. Ringkasan ini juga disimpan ke collection `eda_summary` agar hasil EDA punya artefak tersendiri di MongoDB.

In [ ]:
descriptive_stats = df_clean.select('latitude', 'longitude', 'depth', 'mag', 'depth_log').summary('count', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max')
descriptive_stats.show(truncate=False)

summary_rows = descriptive_stats.collect()
eda_summary_rows = []
for row in summary_rows:
    metric_name = row['summary']
    for feature_name in descriptive_stats.columns:
        if feature_name != 'summary':
            eda_summary_rows.append((feature_name, metric_name, row[feature_name]))

eda_summary_df = spark.createDataFrame(eda_summary_rows, ['feature', 'metric', 'value'])
# eda_summary_df.write.format('mongodb') \
#     .mode('overwrite') \
#     .option('database', MONGO_DB) \
#     .option('collection', MONGO_EDA_SUMMARY_COL) \
#     .save()

print(f'Ringkasan EDA berhasil disimpan ke collection {MONGO_EDA_SUMMARY_COL}.')

time_range = df_clean.selectExpr('min(time) as earliest_time', 'max(time) as latest_time', 'count(*) as total_rows')
time_range.show(truncate=False)

country_counts = df_clean.groupBy('country').agg(count('*').alias('quake_count')).orderBy(col('quake_count').desc()).limit(10)
country_counts.show(truncate=False)

### Tahap 4: Analisis Persebaran dan Distribusi Data
Bagian ini memeriksa pola distribusi gempa melalui frekuensi bulanan, wilayah dengan kejadian tertinggi, serta hubungan awal antara magnitude dan kedalaman.

In [ ]:
numeric_features = ['latitude', 'longitude', 'depth', 'mag', 'x', 'y', 'z', 'depth_log']

stats_query = ' UNION ALL '.join([
    f"SELECT '{feature}' AS feature, round(avg({feature}), 4) AS mean, round(min({feature}), 4) AS minimum, round(max({feature}), 4) AS maximum, round(stddev_samp({feature}), 4) AS stddev FROM earthquake_clean"
    for feature in numeric_features
])

stats_df = spark.sql(stats_query)
stats_df.show(truncate=False)

In [ ]:
monthly_df = spark.sql("""
SELECT date_format(time, 'yyyy-MM') AS month, count(*) AS quake_count
FROM earthquake_clean
GROUP BY date_format(time, 'yyyy-MM')
ORDER BY month
""")

monthly_pdf = monthly_df.toPandas()

fig, ax = plt.subplots()
ax.plot(monthly_pdf['month'], monthly_pdf['quake_count'], marker='o', linewidth=2)
ax.set_title('Tren Jumlah Gempa per Bulan')
ax.set_xlabel('Bulan')
ax.set_ylabel('Jumlah Gempa')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top_region_df = spark.sql("""
SELECT country, count(*) AS quake_count
FROM earthquake_clean
GROUP BY country
ORDER BY quake_count DESC
LIMIT 10
""")

top_region_pdf = top_region_df.toPandas()

fig, ax = plt.subplots()
sns.barplot(data=top_region_pdf, x='quake_count', y='country', ax=ax, palette='Blues_r')
ax.set_title('Top 10 Wilayah dengan Frekuensi Gempa Tertinggi')
ax.set_xlabel('Jumlah Gempa')
ax.set_ylabel('Wilayah')
plt.tight_layout()
plt.show()

In [ ]:
plot_pdf = df_clean.select('mag', 'depth', 'latitude', 'longitude').dropna().sample(False, 0.2, seed=42).limit(5000).toPandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(plot_pdf['mag'], bins=30, kde=True, ax=axes[0, 0], color='#3b82f6')
axes[0, 0].set_title('Distribusi Magnitude')
sns.histplot(plot_pdf['depth'], bins=30, kde=True, ax=axes[0, 1], color='#14b8a6')
axes[0, 1].set_title('Distribusi Kedalaman')
sns.scatterplot(data=plot_pdf, x='mag', y='depth', ax=axes[1, 0], alpha=0.35, s=25, color='#ef4444')
axes[1, 0].set_title('Hubungan Magnitude dan Kedalaman')
corr_matrix = plot_pdf.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='crest', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Heatmap Korelasi')
plt.tight_layout()
plt.show()

### Tahap 5: Persiapan Data untuk Clustering
Fitur numerik disusun kembali ke format vector. Selain itu, kita melakukan **Geographical Undersampling** (maks 5.000 titik per negara) agar klaster tidak bias ke wilayah yang memiliki sensor gempa sangat padat (seperti Amerika Serikat).


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rand

model_df = df_clean.select('time', 'place', 'country', 'latitude', 'longitude', 'depth', 'mag', 'x', 'y', 'z', 'depth_log', 'scaled_features', 'features').dropna(subset=['features'])

# Geographical Undersampling (Max 5000 per country) for training
window_spec = Window.partitionBy('country').orderBy(rand(42))
training_df = model_df.withColumn('rn', row_number().over(window_spec)) \
                      .filter(col('rn') <= 5000) \
                      .drop('rn')

evaluator = ClusteringEvaluator(featuresCol='features', predictionCol='prediction', metricName='silhouette', distanceMeasure='squaredEuclidean')

print(f'Total data keseluruhan (untuk prediksi): {model_df.count()} baris')
print(f'Total data undersampled (untuk training): {training_df.count()} baris')


### Tahap 6: Penentuan Jumlah Cluster Optimal (Elbow Method)
Tahap ini mencari nilai K yang masuk akal sebelum model clustering dilatih dengan menghitung Within-Cluster Sum of Squares (WCSS) dan Silhouette Score.

In [ ]:
k_values = list(range(2, 11))
elbow_rows = []

for k in k_values:
    model = KMeans(k=k, seed=42, featuresCol='features', predictionCol='prediction', maxIter=25).fit(training_df)
    predictions = model.transform(training_df) # Evaluasi siku bisa menggunakan training_df agar lebih cepat
    silhouette = float(evaluator.evaluate(predictions))
    elbow_rows.append((k, float(model.summary.trainingCost), silhouette))

elbow_df = spark.createDataFrame(elbow_rows, ['k', 'wcss', 'silhouette'])
elbow_df.show(truncate=False)

In [ ]:
elbow_pdf = elbow_df.toPandas()

fig, ax1 = plt.subplots()
ax1.plot(elbow_pdf['k'], elbow_pdf['wcss'], marker='o', color='#1f77b4', label='WCSS')
ax1.set_xlabel('Jumlah Cluster (K)')
ax1.set_ylabel('WCSS / Training Cost', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax2 = ax1.twinx()
ax2.plot(elbow_pdf['k'], elbow_pdf['silhouette'], marker='s', color='#d62728', label='Silhouette')
ax2.set_ylabel('Silhouette Score', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')
ax1.set_title('Elbow Method dan Silhouette per Nilai K')
plt.tight_layout()
plt.show()

### Tahap 7: Pelatihan Model (K-Means, Bisecting K-Means dan Gaussian Mixture Model (GMM))
Model clustering dilatih menggunakan data `training_df`, lalu di-apply ke seluruh data `model_df`.


In [ ]:
OPTIMAL_K = 8  # Sesuaikan dengan titik siku pada grafik elbow setelah dijalankan

kmeans = KMeans(k=OPTIMAL_K, seed=42, featuresCol='features', predictionCol='kmeans_cluster', maxIter=25)
kmeans_model = kmeans.fit(training_df)
kmeans_result = kmeans_model.transform(model_df)
kmeans_silhouette = float(ClusteringEvaluator(featuresCol='features', predictionCol='kmeans_cluster', metricName='silhouette', distanceMeasure='squaredEuclidean').evaluate(kmeans_result))

print(f'K-Means Silhouette Score: {kmeans_silhouette:.4f}')
kmeans_result.groupBy('kmeans_cluster').agg(count('*').alias('count'), round(avg('mag'), 4).alias('avg_mag'), round(avg('depth'), 4).alias('avg_depth')).orderBy('kmeans_cluster').show(truncate=False)

In [ ]:
bisecting = BisectingKMeans(k=OPTIMAL_K, seed=42, featuresCol='features', predictionCol='bisect_cluster', maxIter=20)
bisecting_model = bisecting.fit(training_df)
bisecting_result = bisecting_model.transform(model_df)
bisecting_silhouette = float(ClusteringEvaluator(featuresCol='features', predictionCol='bisect_cluster', metricName='silhouette', distanceMeasure='squaredEuclidean').evaluate(bisecting_result))

print(f'Bisecting K-Means Silhouette Score: {bisecting_silhouette:.4f}')
bisecting_result.groupBy('bisect_cluster').agg(count('*').alias('count'), round(avg('mag'), 4).alias('avg_mag'), round(avg('depth'), 4).alias('avg_depth')).orderBy('bisect_cluster').show(truncate=False)

In [ ]:
gmm = GaussianMixture(k=OPTIMAL_K, seed=42, featuresCol='features', predictionCol='gmm_cluster', maxIter=20)
gmm_model = gmm.fit(training_df)
gmm_result = gmm_model.transform(model_df)
# Evaluasi Silhouette untuk GMM (Catatan: GMM adalah soft-clustering, Silhouette adalah pendekatan heuristik di sini)
gmm_silhouette = float(ClusteringEvaluator(featuresCol='features', predictionCol='gmm_cluster', metricName='silhouette', distanceMeasure='squaredEuclidean').evaluate(gmm_result))

print(f'Gaussian Mixture Model Silhouette Score: {gmm_silhouette:.4f}')
gmm_result.groupBy('gmm_cluster').agg(count('*').alias('count'), round(avg('mag'), 4).alias('avg_mag'), round(avg('depth'), 4).alias('avg_depth')).orderBy('gmm_cluster').show(truncate=False)


### Tahap 8: Visualisasi Persebaran Cluster
Plot hasil klaster ketiga model pada peta koordinat spasial (scatter plot *longitude* vs *latitude*).


In [ ]:
kmeans_plot_pdf = kmeans_result.select('longitude', 'latitude', 'kmeans_cluster', 'mag', 'depth').dropna().sample(False, 0.2, seed=42).limit(5000).toPandas()
bisect_plot_pdf = bisecting_result.select('longitude', 'latitude', 'bisect_cluster', 'mag', 'depth').dropna().sample(False, 0.2, seed=42).limit(5000).toPandas()
gmm_plot_pdf = gmm_result.select('longitude', 'latitude', 'gmm_cluster', 'mag', 'depth').dropna().sample(False, 0.2, seed=42).limit(5000).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
sns.scatterplot(data=kmeans_plot_pdf, x='longitude', y='latitude', hue='kmeans_cluster', palette='tab10', s=20, alpha=0.7, ax=axes[0])
axes[0].set_title('K-Means Cluster Map')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

sns.scatterplot(data=bisect_plot_pdf, x='longitude', y='latitude', hue='bisect_cluster', palette='tab10', s=20, alpha=0.7, ax=axes[1])
axes[1].set_title('Bisecting K-Means Cluster Map')
axes[1].set_xlabel('Longitude')

sns.scatterplot(data=gmm_plot_pdf, x='longitude', y='latitude', hue='gmm_cluster', palette='tab10', s=20, alpha=0.7, ax=axes[2])
axes[2].set_title('GMM Cluster Map')
axes[2].set_xlabel('Longitude')

plt.tight_layout()
plt.show()


### Tahap 9: Evaluasi Clustering
Pengukuran Silhouette Score untuk menentukan seberapa cocok titik data dengan clusternya masing-masing. Hasil evaluasi (`model_metrics`) ini di-save ke MongoDB untuk histori metrik evaluasi model cluster.

In [ ]:
comparison_df = spark.createDataFrame([
    ('K-Means', kmeans_silhouette),
    ('Bisecting K-Means', bisecting_silhouette),
    ('Gaussian Mixture Model', gmm_silhouette)
], ['model', 'silhouette'])

comparison_df.show(truncate=False)

model_metrics_df = spark.createDataFrame([
    ('Bisecting K-Means', OPTIMAL_K, bisecting_silhouette),
    ('Gaussian Mixture Model', OPTIMAL_K, gmm_silhouette),
    ('Bisecting K-Means', OPTIMAL_K, bisecting_silhouette)
], ['algorithm', 'k', 'silhouette'])

# model_metrics_df.write.format('mongodb') \
#     .mode('overwrite') \
#     .option('database', MONGO_DB) \
#     .option('collection', MONGO_MODEL_METRICS_COL) \
#     .save()

print(f'Model metrics berhasil disimpan ke collection {MONGO_MODEL_METRICS_COL}.')

### Tahap 10: Penyimpanan Hasil Analisis ke Database
Seluruh label klaster data gempa (`kmeans_cluster`, `bisect_cluster`, `gmm_cluster`) yang dihasilkan akan di-ekstrak beserta kolom aslinya untuk disimpan di MongoDB tanpa fitur tambahan (training). Data inilah yang selanjutnya jadi basis valid untuk proses Visualisasi/Dasboard berikutnya.


In [ ]:
kmeans_export = kmeans_result.select('time', 'place', 'country', 'latitude', 'longitude', 'depth', 'mag', 'kmeans_cluster')
bisecting_export = bisecting_result.select('time', 'place', 'country', 'latitude', 'longitude', 'depth', 'mag', 'bisect_cluster')
gmm_export = gmm_result.select('time', 'place', 'country', 'latitude', 'longitude', 'depth', 'mag', 'gmm_cluster')
MONGO_GMM_COL = os.getenv('MONGO_GMM_COLLECTION', 'gmm_results')

# kmeans_export.write.format('mongodb') \
#     .mode('overwrite') \
#     .option('database', MONGO_DB) \
#     .option('collection', MONGO_KMEANS_COL) \
#     .save()

# bisecting_export.write.format('mongodb') \
#     .mode('overwrite') \
#     .option('database', MONGO_DB) \
#     .option('collection', MONGO_BISECTING_COL) \
#     .save()
# gmm_export.write.format('mongodb') \
#     .mode('overwrite') \
#     .option('database', MONGO_DB) \
#     .option('collection', MONGO_GMM_COL) \
#     .save()

print('Hasil clustering berhasil disimpan ke MongoDB.')